In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/salmaahmedd8/lecttt5555/Structured programing lecture 5.pdf


In [2]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled hugg

In [3]:
pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
NGROK_TOKEN = "3GhNMikAmz9DTZuLcJHTW2dS5Z5_89JpviiRRK3VTxjW2U5KZ"
API_KEY = "secret123"

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_58/2318421842.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def load_llm():

    model_name = "mistralai/Mistral-7B-Instruct-v0.2"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    return tokenizer, model


tokenizer, model = load_llm()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [7]:
def generate_response(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=812,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
def load_pdf(pdf_path):

    loader = PyPDFLoader(pdf_path)

    documents = loader.load()

    return documents


def split_documents(documents):

    text_splitter = CharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )

    chunks = text_splitter.split_documents(documents)

    return chunks


def create_vector_db(chunks):

    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectordb = FAISS.from_documents(
        chunks,
        embedding
    )

    return vectordb



# Load PDF and Build Vector DB


pdf_path = "/kaggle/input/datasets/salmaahmedd8/lecttt5555/Structured programing lecture 5.pdf"

documents = load_pdf(pdf_path)

chunks = split_documents(documents)

vectordb = create_vector_db(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
def ask_pdf(question, vectordb):

    docs = vectordb.similarity_search(question, k=3)

    context = "\n\n".join([doc.page_content for doc in docs])

    rag_prompt = f"""
You are a helpful AI study assistant.

Use ONLY the following context to answer the user's question.

Context:
{context}

Question:
{question}

Answer:
"""

    result = generate_response(rag_prompt)

    return result.split("Answer:")[-1].strip()

In [10]:
question = input("Ask a question: ")

answer = ask_pdf(question, vectordb)

print("\nAnswer:\n")
print(answer)

Ask a question:  what is the lecture about


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer:

The lecture is about the fundamentals of structured programming, specifically focusing on making decisions in programming.


In [11]:
def generate_summary(documents):

    full_text = "\n\n".join([doc.page_content for doc in documents])

    summary_prompt = f"""
You are an AI study assistant.

Read the following document and write a clear, well-organized summary.

Document:
{full_text}

Summary:
"""

    summary = generate_response(summary_prompt)

    return summary.split("Summary:")[-1].strip()

In [12]:
summary = generate_summary(documents)

print(summary)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In this lecture, Dr. Naglaa Fathy discusses the fundamentals of structured programming with a focus on strings and characters. She explains how to work with characters and strings, including string operators, string member functions, and string input. The lecture also covers decision making statements, including if statements, if-else statements, switch statements, relational operators, and logical operators. Examples are provided to illustrate the concepts.


In [29]:
def generate_quiz(documents, num_questions=5):

    full_text = "\n\n".join([doc.page_content for doc in documents])

    quiz_prompt = f"""
You are an AI study assistant.

Create exactly {num_questions} multiple-choice questions based ONLY on the document.

STRICT FORMAT (follow exactly):

1. Question text?

A. Choice 1
B. Choice 2
C. Choice 3
D. Choice 4

Answer: B

2. Question text?

A. Choice 1
B. Choice 2
C. Choice 3
D. Choice 4

Answer: D

Rules:
- EXACTLY 4 choices.
- Put EACH choice on its OWN line.
- NEVER write choices on the same line.
- Leave one blank line before "Answer:".
- Do NOT skip any question.
- Do NOT add explanations.

Document:
{full_text}

Quiz:
"""

    quiz = generate_response(quiz_prompt)

    return quiz.split("Quiz:")[-1].strip()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     41.39.8.123:0 - "POST /quiz HTTP/1.1" 200 OK


In [30]:
quiz = generate_quiz(documents)

print(quiz)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1. Which operator can you use to assign a value to a string?
A) =
B) +
C) &&
D) ||

Answer: A

2. Which operator can you use to join two strings together?
A) +
B) =
C) &&
D) ||

Answer: A

3. How do you assign the entire string str1 to str2?
A) str1 = str2;
B) str1 += str2;
C) str1.assign(str2);
D) str2.assign(str1);

Answer: C

4. Which function can you use to read in a string that may contain blanks?
A) cin >>
B) cin.getline();
C) cin.ignore();
D) cin.get();

Answer: B

5. Which operator can you use to create relational expressions from other relational expressions?
A) &&
B) ||
C) >
D) <

Answer: A

6. If the number is 10, what will the value of x be?
int x;
int y = 100;
if (y == 100) {
    x = 1;
} else {
    x = 0;
}

Answer: The value of x will be 1.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     41.39.8.123:0 - "POST /quiz HTTP/1.1" 200 OK


In [15]:
def generate_notes(documents):

    full_text = "\n\n".join([doc.page_content for doc in documents])

    notes_prompt = f"""
You are an AI study assistant.

Read the following document and create clear study notes.

Rules:
- Organize the notes using headings.
- Use bullet points.
- Highlight the most important concepts.
- Keep the notes concise and easy to review.

Document:
{full_text}

Study Notes:
"""

    notes = generate_response(notes_prompt)

    return notes.split("Study Notes:")[-1].strip()

In [16]:
notes = generate_notes(documents)

print(notes)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


**Fundamentals of Structured Programming**

## Lecture 5

### String

- A `char` holds a single character.
- A `string` holds a sequence of characters.
- Both can be used in assignment statements.
- Both can be displayed with `cout` and `<<`.

### String Operators

- `=` assigns a value to a string.
- `+` joins two strings together.
- `+=` concatenates a string onto the end of another one.

### String Member Functions

- `length()` – the number of characters in a string.

### String Assignments

- `assign()` – put repeated characters in a string. Can be used for formatting output.
- `assign(str1, start, length)` assigns a substring of length from string `str1` starting at index `start` to the string.
- `assign(n, ch)` assigns `n` occurrences of character `ch` to the string.

### String Input

- Reading in a string object: `string name; cin >> name;`.
- Using `cin` with the `>>` operator to input strings can cause problems: it passes over and ignores any leading whitespace characters.



In [17]:
def generate_study_plan(documents, days, hours_per_day):

    full_text = "\n\n".join([doc.page_content for doc in documents])

    planner_prompt = f"""
You are an AI study assistant.

Create a study plan based ONLY on the following document.

Requirements:
- Exam is after {days} days.
- Student studies {hours_per_day} hours per day.
- Divide the topics across the available days.
- Include revision before the exam.
- Make the plan realistic and organized.

Document:
{full_text}

Study Plan:
"""

    plan = generate_response(planner_prompt)

    return plan.split("Study Plan:")[-1].strip()

In [18]:
plan = generate_study_plan(
    documents,
    days=7,
    hours_per_day=2
)

print(plan)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Day 1:
- Read and understand the document about strings.
- Practice using string operators and member functions.

Day 2:
- Learn about string input using cin and getline.
- Understand the difference between char and string input.

Day 3:
- Learn about decision making statements: if, if-else, switch, and conditional operator.
- Understand relational and logical operators.

Day 4:
- Practice using nested if statements and if-else if statements.
- Review decision making concepts.

Day 5:
- Learn about menu-driven programs and their organization.

Day 6:
- Review all topics covered so far.
- Practice solving problems related to strings, decision making, and menu-driven programs.

Day 7:
- Revise all topics covered in the previous days.
- Take practice exams or quizzes to assess understanding.


In [33]:
from fastapi import FastAPI, Request, HTTPException, UploadFile, File

app = FastAPI()

In [34]:
from fastapi import UploadFile, File

vectordb = None

@app.post("/upload")
async def upload_pdf(
    req: Request,
    file: UploadFile = File(...)
):

    global vectordb, documents

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    if file.content_type != "application/pdf":
        raise HTTPException(
            status_code=400,
            detail="Please upload a PDF file."
        )

    pdf_path = f"/tmp/{file.filename}"

    with open(pdf_path, "wb") as f:
        f.write(await file.read())

    documents = load_pdf(pdf_path)

    chunks = split_documents(documents)

    vectordb = create_vector_db(chunks)

    return {
        "message": "PDF uploaded successfully."
    }

In [35]:
@app.post("/chat")
async def chat(req: Request):

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    if vectordb is None:
        raise HTTPException(
            status_code=400,
            detail="Upload a PDF first."
        )

    data = await req.json()

    question = data.get("question", "")

    if not question:
        raise HTTPException(
            status_code=400,
            detail="Question is required."
        )

    answer = ask_pdf(question, vectordb)

    return {
        "answer": answer
    }

In [36]:
@app.post("/summary")
async def summary(req: Request):

    global documents

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    if documents is None:
        raise HTTPException(
            status_code=400,
            detail="Upload a PDF first."
        )

    print(type(documents))

    summary = generate_summary(documents)

    return {
        "summary": summary
    }

In [37]:
@app.post("/quiz")
async def quiz(req: Request):
    global documents  

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    if documents is None:
        raise HTTPException(status_code=400, detail="Upload a PDF first.")

    data = await req.json()
    num_questions = data.get("num_questions", 5)

    
    quiz_res = generate_quiz(documents, num_questions)

    return {"quiz": quiz_res}

In [38]:
@app.post("/notes")
async def notes(req: Request):
    global documents  

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    if documents is None:
        raise HTTPException(status_code=400, detail="Upload a PDF first.")

    
    notes_res = generate_notes(documents)

    return {"notes": notes_res}

In [39]:
@app.post("/planner")
async def planner(req: Request):
    global documents

    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    if documents is None:
        raise HTTPException(status_code=400, detail="Upload a PDF first.")

    data = await req.json()
    days = data.get("days", 7)
    hours_per_day = data.get("hours_per_day", 2)

    
    plan_res = generate_study_plan(documents, days, hours_per_day)

    return {"plan": plan_res}

In [41]:
import threading
import time
import socket
import uvicorn

from fastapi import FastAPI, Request, HTTPException
from pyngrok import ngrok

In [42]:
from pyngrok import ngrok, conf
import socket
import threading
import time
import uvicorn
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = "3GhNMikAmz9DTZuLcJHTW2dS5Z5_89JpviiRRK3VTxjW2U5KZ"
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://volatile-rival-dreadful.ngrok-free.dev


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:40743 (Press CTRL+C to quit)


INFO:     41.39.8.123:0 - "POST /upload HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     41.39.8.123:0 - "POST /quiz HTTP/1.1" 200 OK
INFO:     41.39.8.123:0 - "POST /upload HTTP/1.1" 200 OK


...

INFO:     41.39.8.123:0 - "POST /quiz HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     41.39.8.123:0 - "POST /notes HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     41.39.8.123:0 - "POST /planner HTTP/1.1" 200 OK
